In [0]:
import pandas as pd



In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_periodo (
 periodo_id DATE,
 dia INT,
 mes INT,
 `año` INT
)
""")


In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_periodo")
spark.sql("""
CREATE TABLE workspace.gold.dim_periodo (
 fecha_id INT,
 periodo DATE,
 `año` INT,
 mes INT,
 nombre_mes STRING,
 trimestre INT
)
""")

df = spark.table("internet_fijo_elt.silver.conexiones_internet_fijo")
from pyspark.sql import functions as F

dim_fecha = (
    df
    .select("periodo")
    .distinct()
    .withColumn("fecha_id", F.date_format("periodo", "yyyyMM").cast("int"))
    .withColumn("año", F.year("periodo"))
    .withColumn("mes", F.month("periodo"))
    .withColumn("trimestre", F.quarter("periodo"))
    .withColumn("nombre_mes", F.date_format("periodo", "MMMM"))
    .select("fecha_id", "periodo", "año", "mes", "nombre_mes", "trimestre")
)

dim_fecha.write.mode("append").saveAsTable("workspace.gold.dim_periodo")